In [2]:
import torch
from torch.utils.data import DataLoader, TensorDataset

import sys, os
sys.path.append(os.path.expanduser("~/Desktop/projects/audio-filler"))

from models.model2.model2 import AudioEncoder1

In [3]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# Dummy audio data (batch of 8 mono waveforms, each 240000 samples long)
x_dummy = torch.randn(8, 1, 240000)

# Dummy class labels (15 classes)
y_dummy = torch.randint(0, 15, (8,))

# DataLoader setup
train_loader = DataLoader(TensorDataset(x_dummy, y_dummy), batch_size=2, shuffle=True)


In [4]:
model = AudioEncoder1()
model.eval()

AudioEncoder1(
  (tanh): Tanh()
  (leaky_relu): LeakyReLU(negative_slope=0.2)
  (conv1_1): Conv1d(3, 32, kernel_size=(8,), stride=(4,), padding=(2,))
  (bn32): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv1_2): Conv1d(32, 128, kernel_size=(5,), stride=(3,), padding=(1,))
  (bn128): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2_1): Conv1d(3, 32, kernel_size=(5,), stride=(3,), padding=(1,))
  (conv2_2): Conv1d(32, 64, kernel_size=(4,), stride=(2,), padding=(1,))
  (bn64): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2_3): Conv1d(64, 128, kernel_size=(4,), stride=(2,), padding=(1,))
  (final_conv_1): Conv1d(256, 128, kernel_size=(4,), stride=(2,), padding=(1,))
  (final_bn128): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (final_conv2): Conv1d(128, 128, kernel_size=(4,), stride=(2,), padding=(1,))
  (final_conv3): Conv1d(128,

In [5]:
import torch
import torch.optim as optim
import pandas as pd
import os
import torch.nn.functional as F

def train_model(model, train_loader, num_epochs=10, lr=1e-4, device="cuda",
                csv_path="training_log.csv", save_dir="checkpoints"):

    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # Create save directory
    os.makedirs(save_dir, exist_ok=True)

    # Prepare CSV logging file
    if not os.path.exists(csv_path):
        pd.DataFrame(columns=["epoch", "batch", "total_loss", "class_loss", "recon_loss",
                              "kl_div", "class_acc", "recon_acc"]).to_csv(csv_path, index=False)

    for epoch in range(num_epochs):
        model.train()
        for batch_idx, (x, labels) in enumerate(train_loader):
            x, labels = x.to(device), labels.to(device)

            optimizer.zero_grad()
            class_logits, spec_recon, mu, log_var = model(x)
            total_loss, class_loss, recon_loss, kl_div = model.loss_function(
                class_logits, spec_recon, mu, log_var, x, labels
            )

            total_loss.backward()
            optimizer.step()

            # --- Metrics ---
            preds = class_logits.argmax(dim=1)
            class_acc = (preds == labels).float().mean().item()

            spec_target = model.recon_to_spec(x).unsqueeze(1)
            recon_acc = 1.0 - F.mse_loss(spec_recon, spec_target).item()

            # Log results to CSV
            log_entry = {
                "epoch": epoch + 1,
                "batch": batch_idx + 1,
                "total_loss": total_loss.item(),
                "class_loss": class_loss.item(),
                "recon_loss": recon_loss.item(),
                "kl_div": kl_div.item(),
                "class_acc": class_acc,
                "recon_acc": recon_acc
            }
            pd.DataFrame([log_entry]).to_csv(csv_path, mode="a", header=False, index=False)

            
            print(f"Epoch {epoch+1} Batch {batch_idx+1}: "
                    f"Loss={total_loss.item():.4f}, "
                    f"ClsAcc={class_acc:.4f}, ReconAcc={recon_acc:.4f}")

        # ---- Save model after each epoch ----
        checkpoint_path = os.path.join(save_dir, f"model_epoch_{epoch+1}.pt")
        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
        }, checkpoint_path)

        print(f"✅ Saved checkpoint: {checkpoint_path}")


In [6]:
train_model(model, train_loader, num_epochs=10, lr=1e-4, device="mps",
            csv_path="training_log.csv", save_dir="checkpoints")

RuntimeError: Given groups=1, weight of size [128, 251, 2], expected input[2, 126, 51] to have 251 channels, but got 126 channels instead